# Day 2 · Lab 3 — Notebook Log Analysis

**AI for Cybersecurity Professionals · Day 2: AI for Defense**

In **Lab 2** you opened the raw log and looked at it *by eye* — you saw the fields and got a
feel for what a login record contains. That approach works for a handful of lines, but a real
server writes millions a day. In this lab we do the same inspection **systematically, in code**,
so it scales. This same dataset feeds every later Day 2 lab (rule-based parsing, AI triage,
anomaly detection), so it is worth understanding well.

### What is this data? (recap from Lab 1)
It is a **login (authentication) log** — one row every time someone tried to sign in to a
web application: *when* they tried, *from what IP address and country*, *what device and
browser*, and *whether the login succeeded*.

### What you'll do
1. Load the log into a table (a pandas *DataFrame*).
2. Look at its shape and columns.
3. Count successful vs. failed logins.
4. Find **brute-force** attacks (one IP failing over and over).
5. Find a **distributed** attack (one victim hammered from many countries).
6. Find the needle: **account takeovers** (an attacker who actually got in).

> No prior Python or Linux experience needed — every line of code is explained in comments.


## Step 1 — Set up our tools

Python code lives in *cells*. Click a cell and press **▶** (or **Shift+Enter**) to run it.

We import two libraries:
- **pandas** — think of it as *Excel for Python*. It loads tables and lets us filter, count,
  and group rows. We nickname it `pd` so we type less.
- **matplotlib** — a plotting library, nicknamed `plt`, for simple charts.


In [ ]:
# 'import' loads a library so we can use it.
# 'as pd' gives it a short nickname.
import pandas as pd
import matplotlib.pyplot as plt

# These two lines just make tables and charts easier to read.
pd.set_option("display.max_columns", 30)   # show all columns, don't hide any
pd.set_option("display.width", 160)         # use a wide layout
print("Libraries loaded. Ready to go.")

## Step 2 — Load the log file

The log is a **CSV** file (Comma-Separated Values — a plain-text spreadsheet) named
`day2_auth_logs.csv`.

The cell below is written to work **three ways** automatically:
- If your instructor gave you a web link, paste it into `DATA_URL` and it loads from there.
- Otherwise, if the file is already next to the notebook, it loads that.
- Otherwise (typical in Google Colab), it will pop up a **file chooser** so you can upload it.

You normally don't need to change anything — just run it.


In [ ]:
# ----- Where to find the data -------------------------------------------------
DATA_FILE = "day2_auth_logs.csv"   # the file name we expect
DATA_URL  = ""                      # optional: instructor may paste a raw CSV link here

import os

def load_logs():
    """Load the login log no matter where it lives (URL, local file, or Colab upload)."""
    # parse_dates=[...] tells pandas the 'Login Timestamp' column holds real dates/times,
    # not just text. That lets us do time math later (e.g. "5 failures in 60 seconds").
    if DATA_URL:                                   # 1) a web link was provided
        return pd.read_csv(DATA_URL, parse_dates=["Login Timestamp"])
    for path in [DATA_FILE, "data/" + DATA_FILE]:  # 2) the file sits near the notebook
        if os.path.exists(path):
            return pd.read_csv(path, parse_dates=["Login Timestamp"])
    from google.colab import files                 # 3) Colab: ask the user to upload it
    uploaded = files.upload()
    filename = list(uploaded.keys())[0]
    return pd.read_csv(filename, parse_dates=["Login Timestamp"])

df = load_logs()                                    # 'df' (DataFrame) is our table of logs
print("Loaded", len(df), "log entries.")

## Step 3 — First look at the data

`df.shape` tells us **(rows, columns)**. `df.head()` shows the first few rows so we can
see what a login record looks like.


In [ ]:
# .shape is (number_of_rows, number_of_columns)
print("Rows and columns:", df.shape)

# .head() shows the first 5 rows. Great for a quick peek.
df.head()

### What do the columns mean?

| Column | Plain-English meaning |
|---|---|
| `Login Timestamp` | When the login was attempted |
| `Username` / `User ID` | Which account was being accessed |
| `IP Address` | The internet address the attempt came from |
| `Country` / `Region` / `City` | Where that IP is located |
| `ASN` | The network operator that owns the IP (a data-centre vs. a home ISP) |
| `User Agent String` | What browser/app made the request (a real browser vs. a script) |
| `Device Type` | desktop / mobile / tablet |
| `Login Successful` | **True** = they got in, **False** = wrong password etc. |
| `Is Attack IP` | The IP is on a known-bad list |
| `Is Account Takeover` | **Confirmed** compromise — the attacker actually got in |

> ⚠️ **Important — these last two columns are an *answer key*, not log data.**
> A real login server never records `Is Attack IP` or `Is Account Takeover` — those labels
> were added *after* the fact by analysts who already knew the outcome. In this course we
> use them **only so you can check your work and understand what an attack looks like while
> reading the log by eye.** They are **never** used inside any detection logic. In Labs 2–5,
> the rules and machine-learning models must find attacks *without* looking at these two
> columns — otherwise we'd just be reading the answer off the back of the book. The only
> genuine outcome field is `Login Successful`, which every real auth server does record.


In [ ]:
# .info() lists every column, how many values are filled in, and each column's type.
df.info()

## Step 4 — How many logins succeeded vs. failed?

`value_counts()` counts how many times each value appears in a column. On the
`Login Successful` column it tells us successes vs. failures.


In [ ]:
# Count True (success) vs False (failure)
counts = df["Login Successful"].value_counts()
print(counts)

# The same numbers as a simple bar chart.
counts.plot(kind="bar", color=["#2e7d32", "#c62828"])
plt.title("Successful vs. Failed Logins")
plt.xlabel("Login Successful")
plt.ylabel("Number of attempts")
plt.xticks(rotation=0)
plt.show()

A large pile of **failures** is our first clue. Legitimate users occasionally fat-finger a
password, but thousands of failures usually means someone (or some *bot*) is guessing.
Let's find out **who**.


## Step 5 — Find brute-force attackers (one IP, many failures)

A **brute-force** attack is one source trying password after password. So we:
1. Keep only the **failed** logins.
2. Group them **by IP address**.
3. Count each IP's failures and show the worst offenders.


In [ ]:
# Step 1: keep only rows where the login FAILED.
#   df["Login Successful"] == False  ->  a column of True/False
#   df[ ... ]                         ->  keep only the rows that are True
failed = df[df["Login Successful"] == False]
print("Total failed logins:", len(failed))

# Step 2 & 3: count failures per IP, biggest first, show the top 10.
top_bad_ips = failed["IP Address"].value_counts().head(10)
print("\nTop 10 IPs by number of FAILED logins:")
print(top_bad_ips)

The `10.3.205.x` addresses are an **internal subnet** hammering one account — a classic
brute-force burst. Below we confirm it's fast (many tries in seconds), which is exactly the
pattern the rule-based lab (Lab 2) will detect with *"more than 5 failures from one IP in
60 seconds."*


In [ ]:
# Pick the single worst IP and look at how tightly packed its attempts are in time.
worst_ip = top_bad_ips.index[0]
attempts = df[df["IP Address"] == worst_ip].sort_values("Login Timestamp")

print("Worst IP:", worst_ip)
print("Number of attempts:", len(attempts))
print("First attempt:", attempts["Login Timestamp"].min())
print("Last attempt: ", attempts["Login Timestamp"].max())

# How long did the whole burst take? (.max - .min gives a time span)
span = attempts["Login Timestamp"].max() - attempts["Login Timestamp"].min()
print("Whole burst lasted:", span)

## Step 6 — Find a distributed attack (one victim, many countries)

Attackers also do the opposite: hammer **one account** from **hundreds of IPs** worldwide
(this is *credential stuffing*). Let's find the single most-attacked account.


In [ ]:
# Which USER has the most failed logins?
top_victims = failed["User ID"].value_counts().head(5)
print("Accounts with the most failed logins:")
print(top_victims)

# Zoom in on the #1 victim.
victim_id = top_victims.index[0]
victim_rows = df[df["User ID"] == victim_id]

print("\nMost-attacked account:", victim_id)
print("Total attempts against it:", len(victim_rows))
print("Distinct IPs used:      ", victim_rows["IP Address"].nunique())
print("Distinct countries:     ", victim_rows["Country"].nunique())
print("Successful logins:      ", (victim_rows["Login Successful"] == True).sum())

Hundreds of attempts from dozens of countries, and **zero** successes — the attackers
never guessed this password. Good. But did anyone get in somewhere else? That's the needle.


## Step 7 — Find the needle: account takeovers

The most dangerous event is a login that **succeeded from a known-bad IP**. The dataset
marks *confirmed* compromises in the `Is Account Takeover` column. Let's pull those out.


In [ ]:
# Keep rows where Is Account Takeover is True. These are the confirmed break-ins.
takeovers = df[df["Is Account Takeover"] == True]
print("Confirmed account takeovers:", len(takeovers))

# Show the useful columns for each one.
takeovers[["Login Timestamp", "Username", "IP Address", "Country",
           "Device Type", "User Agent String", "Login Successful"]]

Notice the pattern in each takeover:
- The login **succeeded** (`Login Successful = True`)...
- ...from a **foreign country** the victim normally never logs in from...
- ...using a **script**, not a real browser (`python-requests`, `curl`, `Go-http-client`).

Let's prove the "foreign" part by comparing a victim's takeover country to where they
*usually* log in from.


In [ ]:
# Take the first takeover and compare it to the victim's normal logins.
one = takeovers.iloc[0]                       # .iloc[0] = the first row
victim = one["User ID"]

# The victim's NON-attack logins (their normal behaviour).
normal = df[(df["User ID"] == victim) & (df["Is Attack IP"] == False)]

print("Victim:", one["Username"])
print("Takeover came from:", one["IP Address"], "(", one["Country"], ")")
print("But victim normally logs in from:", list(normal["Country"].unique()))
print("...on device(s):", list(normal["Device Type"].unique()))

## Step 8 — A quick map of where logins come from

Grouping by country shows the geographic spread. Most traffic is from one home country;
the long tail of one-off countries is where a lot of attack traffic hides.


In [ ]:
# Top 10 source countries by number of login attempts.
top_countries = df["Country"].value_counts().head(10)
print(top_countries)

top_countries.plot(kind="bar", color="#1565c0")
plt.title("Top 10 Source Countries (all login attempts)")
plt.xlabel("Country")
plt.ylabel("Number of attempts")
plt.xticks(rotation=0)
plt.show()

## Wrap-up

You have loaded the course security log and, with a few lines of pandas, surfaced:

- **Brute force** — one IP failing many times fast (`10.3.205.x`).
- **Distributed credential stuffing** — one victim hit from dozens of countries.
- **Account takeover** — the needle: a script logging in successfully from abroad.

Everything you found here was **rule-of-thumb** detection done by hand in code. In the labs
that follow we will (**Lab 4**) ask an **AI model** to triage the same log, and (**Lab 5**)
train a **machine-learning model** to find anomalies **without** being told the rules — then
compare which approach wins.

**This exact dataset (`day2_auth_logs.csv`) feeds every one of those labs**, so keep it handy.

### Try it yourself
1. Change `.head(10)` to `.head(20)` in Step 5 — do new suspicious IPs appear?
2. In Step 8, filter to only failed logins before counting countries. Does the ranking change?
3. Find every login that happened between midnight and 5 AM. *Hint:* `df["Login Timestamp"].dt.hour`.
